In [30]:
import numpy as np
import pandas as pd
import skimage.io
import skimage.measure
import os
import xml.etree.ElementTree as ET
import ipywidgets as widgets
from IPython.display import display


### ---- UI Elements ---- ###
# File selection
input_folder_widget = widgets.Text(value="path/to/your/dataset", description="Input Folder:")

# Data type selection
data_type_selection = widgets.Dropdown(
    options=["User provided points", "Segmentation mask", "Tracking result"],
    value="User provided points", description="Data Type:"
)

# Tracking format selection (only visible if "Tracking result" is chosen)
tracking_format_selection = widgets.Dropdown(
    options=["TrackMate (CSV)", "TrackMate (XML)", "Mastodon/Elephant", "Ultrack"],
    value="TrackMate (CSV)", description="Tracking Format:",
    disabled=True
)

# Run button
run_button = widgets.Button(description="Run Transformation", button_style="success")
output_log = widgets.Output()

# Title and introduction
title = widgets.HTML(
    value="<h1 style='text-align:center; color:#336699;'>📌 Coordinate Transformation Tool</h1>",
)

description = widgets.HTML(
    value="""
    <p><strong>Welcome to the Coordinate Transformation Tool!</strong></p>
    <p>This tool allows you to transform coordinates from an original reference frame into a new one.</p>
    <ul>
        <li>📍 <b>User Provided Points</b>: Transform manually provided points stored in CSV files.</li>
        <li>🧩 <b>Segmentation Mask</b>: Extract and transform centroids from segmented cell images.</li>
        <li>📊 <b>Tracking Result</b>: Convert tracking data from TrackMate, Mastodon, or Ultrack.</li>
    </ul>
    <p>Select the appropriate data type and start processing!</p>
    """,
)

data_type_help = widgets.HTML(
    value="""
    <p><b>🛠️ Instructions:</b></p>
    <ul>
        <li><b>User Provided Points:</b> Select this option if you have manually provided coordinates stored in a CSV file. Each file should be named <code>image_name_points.csv</code>.</li>
        <li><b>Segmentation Mask:</b> Use this if you have images with segmentation masks and need to extract and transform cell centroids. Files should follow the <code>image_name_mask.tif</code> format.</li>
        <li><b>Tracking Result:</b> Select this if you are working with tracking data from TrackMate, Mastodon, or Ultrack. Choose the correct format (CSV or XML) when using TrackMate.</li>
            <ul>
                <li><b>TrackMate:</b> Transform TrackMate tracking coordinates for a 3D+t acquisition. The folder should contain at least the reference frame csv named as imagename_at_t0.csv and the TrackMate result named as imagename_at_t0_spots.csv or imagename_at_<t0-tfinal>.xml.</li>
                <li><b>Mastodon/Elephant:</b> Same as for TrackMate. ⚠️ Before you start, rename your Mastodon spot file as "yourimagename_at_t0"-Spot.csv.</li>
                <li><b>Ultrack:</b> Same as for TrackMate. ⚠️ Before you start, rename your Ultrack spot file as "yourimagename_at_t0"-track.csv</li>
            </ul>
    </ul>
    """,
)

# UI layout
ui = widgets.VBox([title, description, data_type_help, input_folder_widget, data_type_selection, tracking_format_selection, run_button, output_log])


### ---- Core Functions ---- ###
def load_coordinate_system(csv_path):
    """Loads origin and basis vectors from a reference CSV."""
    df = pd.read_csv(csv_path, header=None)
    origin = np.array(list(map(float, df.iloc[0, 1].strip("[]").split())))
    u = np.array(list(map(float, df.iloc[1, 1].strip("[]").split())))
    v = np.array(list(map(float, df.iloc[2, 1].strip("[]").split())))
    w = np.array(list(map(float, df.iloc[3, 1].strip("[]").split())))
    return origin, u, v, w

def transform_points(points, origin, u, v, w):
    """Transforms a set of points into the new coordinate system."""
    transformation_matrix = np.column_stack((u, v, w))
    return np.linalg.solve(transformation_matrix, (points - origin).T).T

# SECTION 1

def transform_points_csv(input_folder):
    """
    Process all CSV files containing point coordinates in the input folder and transform them.
    - Detects corresponding reference CSV files.
    - Updates the original CSV with transformed coordinates.

    Parameters:
    - input_folder: Path to the dataset containing point CSVs and reference coordinate CSVs.
    """
    files = os.listdir(input_folder)
    point_files = [f for f in files if f.endswith("_points.csv")]

    for point_file in point_files:
        base_name = point_file.replace("_points.csv", "")
        points_csv_path = os.path.join(input_folder, point_file)
        ref_csv_path = os.path.join(input_folder, base_name + ".csv")

        # Ensure the reference CSV exists
        if not os.path.exists(ref_csv_path):
            print(f"⚠️ Missing reference file for {base_name}, skipping.")
            continue

        # Load the reference coordinate system
        origin, u, v, w = load_coordinate_system(ref_csv_path)

        # Load the points CSV
        df = pd.read_csv(points_csv_path)

        # Ensure the required columns exist
        if not {"Z", "Y", "X"}.issubset(df.columns):
            print(f"⚠️ Skipping {point_file}: Missing required columns (Z, Y, X).")
            continue

        points = df[["Z", "Y", "X"]].values

        # Transform the points
        transformed_points = transform_points(points, origin, u, v, w)

        # Add new columns to the original DataFrame
        df[["X_new", "Y_new", "Z_new"]] = transformed_points

        # Save the updated CSV
        df.to_csv(points_csv_path, index=False)
        print(f"✅ Transformed points saved in {points_csv_path}")


# SECTION 2

def process_segmentation_folder(input_folder):
    """
    Process all segmentation masks in the input folder and transform the centroids.
    - Detects corresponding reference CSV files.
    - Saves the transformed centroids in a structured folder.

    Parameters:
    - input_folder: Path to the dataset containing images, masks, and reference CSVs.
    """

    # Get all segmentation masks in the folder
    mask_files = [f for f in os.listdir(input_folder) if f.endswith("_mask.tif")]

    for mask_file in mask_files:
        base_name = mask_file.replace("_mask.tif", "")
        mask_path = os.path.join(input_folder, mask_file)
        ref_csv_path = os.path.join(input_folder, base_name + ".csv")

        # Ensure the reference CSV exists
        if not os.path.exists(ref_csv_path):
            print(f"⚠️ Missing reference file for {base_name}, skipping.")
            continue

        # Load segmentation mask
        mask = skimage.io.imread(mask_path)

        # Extract cell centroids using regionprops
        props = skimage.measure.regionprops(mask)
        centroids = np.array([prop.centroid for prop in props])

        # Load the reference coordinate system
        origin, u, v, w = load_coordinate_system(ref_csv_path)

        # Transform centroids
        transformed_centroids = transform_points(centroids, origin, u, v, w)

        # Save the transformed centroids
        output_csv_path = os.path.join(input_folder, base_name + "_cells_coordinates.csv")
        pd.DataFrame(transformed_centroids, columns=["X_new", "Y_new", "Z_new"]).to_csv(output_csv_path, index=False)

        print(f"✅ Transformed centroids saved: {output_csv_path}")

# SECTION 3

def transform_trackmate_res(input_folder, format="csv"):
    """
    Process TrackMate CSV file containing spots coordinates across time.

    Parameters:
    - input_folder: Path to the dataset containing point CSVs and reference coordinate CSVs.
    """
    files = os.listdir(input_folder)
    if format == "csv":
        point_files = [f for f in files if f.endswith("_spots.csv")]

        for point_file in point_files:
            base_name = point_file.replace("_spots.csv", "")
            tracks_csv_path = os.path.join(input_folder, point_file)
            ref_csv_path = os.path.join(input_folder, base_name + ".csv")

            # Ensure the reference CSV exists
            if not os.path.exists(os.path.join(input_folder, ref_csv_path)):
                print(f"⚠️ Missing reference file for {base_name}, skipping.")
                continue

            # Load the reference coordinate system
            origin, u, v, w = load_coordinate_system(ref_csv_path)

            # Load the points CSV
            df = pd.read_csv(tracks_csv_path)

            # Extract point coordinates (Z, Y, X) and reorder them to (X, Y, Z) for consistency
            points = df[["POSITION_Z", "POSITION_Y", "POSITION_X"]].values[3:, :].astype("float32")

            # Transform the points
            transformed_points = transform_points(points, origin, u, v, w)

            header = np.array([["X", "Y", "Z"], ["X", "Y", "Z"], ["(micron)", "(micron)", "(micron)"]])

            # Add new columns to the original DataFrame
            df[["NEW_POSITION_X", "NEW_POSITION_Y", "NEW_POSITION_Z"]] = np.vstack((header, transformed_points))

            # Save the updated CSV
            df.to_csv(tracks_csv_path, index=False)
            print(f"✅ Transformed points saved in {tracks_csv_path}")
    else:
        xml_files = [f for f in files if f.endswith(".xml")]

        for xml_file in xml_files:
            base_name = xml_file.replace(".xml", "")
            xml_path = os.path.join(input_folder, xml_file)
            first_image_index = xml_file.split("<")[1].split("-")[0]
            ref_csv_path = os.path.join(input_folder, xml_file.split("<")[0] + first_image_index + ".csv")

            # Ensure the reference CSV exists
            if not os.path.exists(ref_csv_path):
                print(f"⚠️ Missing reference file for {base_name}, skipping.")
                continue

            # Load the reference coordinate system
            origin, u, v, w = load_coordinate_system(ref_csv_path)

            # Parse the XML file
            tree = ET.parse(xml_path)
            root = tree.getroot()

            # Find all spot elements
            spots = root.findall(".//Spot")

            # Extract original coordinates
            original_points = []
            for spot in spots:
                x = float(spot.attrib["POSITION_X"])
                y = float(spot.attrib["POSITION_Y"])
                z = float(spot.attrib["POSITION_Z"])
                original_points.append([z, y, x])

            # Convert to NumPy array
            original_points = np.array(original_points)

            # Transform the points
            transformed_points = transform_points(original_points, origin, u, v, w)

            # Update XML with new transformed coordinates
            for spot, new_coords in zip(spots, transformed_points):
                spot.set("NEW_POSITION_X", str(new_coords[0]))
                spot.set("NEW_POSITION_Y", str(new_coords[1]))
                spot.set("NEW_POSITION_Z", str(new_coords[2]))

           # Overwrite the original XML file
            tree.write(xml_path, encoding="utf-8", xml_declaration=True)

            print(f"✅ Transformed XML updated in place: {xml_path}")

def transform_mastodon_csv(input_folder):
    """
    Process Mastodon CSV file containing spots coordinates across time.

    Parameters:
    - input_folder: Path to the dataset containing point CSVs and reference coordinate CSVs.
    """
    files = os.listdir(input_folder)

    point_files = [f for f in files if f.endswith("-Spot.csv")]

    for point_file in point_files:
        base_name = point_file.replace("-Spot.csv", "")
        tracks_csv_path = os.path.join(input_folder, point_file)
        ref_csv_path = os.path.join(input_folder, base_name + ".csv")

        # Ensure the reference CSV exists
        if not os.path.exists(os.path.join(input_folder, ref_csv_path)):
            print(f"⚠️ Missing reference file for {base_name}, skipping.")
            continue

        # Load the reference coordinate system
        origin, u, v, w = load_coordinate_system(ref_csv_path)

        # Load the points CSV
        df = pd.read_csv(tracks_csv_path)

        # Extract point coordinates (Z, Y, X) and reorder them to (X, Y, Z) for consistency
        points = df[["Spot position.2", "Spot position.1", "Spot position"]].values[2:, :].astype("float32")

        # Transform the points
        transformed_points = transform_points(points, origin, u, v, w)

        header = np.array([["NEW_POSITION_X", "NEW_POSITION_Y", "NEW_POSITION_Z"], ["(micron)", "(micron)", "(micron)"]])

        # Add new columns to the original DataFrame
        df[["New Spot position", "New Spot position.1", "New Spot position.2"]] = np.vstack((header, transformed_points))

        # Save the updated CSV
        df.to_csv(tracks_csv_path, index=False)
        print(f"✅ Transformed points saved in {tracks_csv_path}")

def transform_ultrack_csv(input_folder):
    """
    Process Ultrack CSV file containing spots coordinates across time.

    Parameters:
    - input_folder: Path to the dataset containing point CSVs and reference coordinate CSVs.
    """
    files = os.listdir(input_folder)

    point_files = [f for f in files if f.endswith("-track.csv")]

    for point_file in point_files:
        base_name = point_file.replace("-track.csv", "")
        tracks_csv_path = os.path.join(input_folder, point_file)
        ref_csv_path = os.path.join(input_folder, base_name + ".csv")

        # Ensure the reference CSV exists
        if not os.path.exists(os.path.join(input_folder, ref_csv_path)):
            print(f"⚠️ Missing reference file for {base_name}, skipping.")
            continue

        # Load the reference coordinate system
        origin, u, v, w = load_coordinate_system(ref_csv_path)

        # Load the points CSV
        df = pd.read_csv(tracks_csv_path)

        # Extract point coordinates (Z, Y, X) and reorder them to (X, Y, Z) for consistency
        points = df[["z", "y", "x"]].values.astype("float32")

        # Transform the points
        transformed_points = transform_points(points, origin, u, v, w)

        # Add new columns to the original DataFrame
        df[["x_new", "y_new", "z_new"]] = transformed_points

        # Save the updated CSV
        df.to_csv(tracks_csv_path, index=False)
        print(f"✅ Transformed points saved in {tracks_csv_path}")

def update_tracking_format_visibility(change):
    """Show/hide tracking format selection based on the chosen data type."""
    if change['new'] == "Tracking result":
        tracking_format_selection.disabled = False
    else:
        tracking_format_selection.disabled = True

data_type_selection.observe(update_tracking_format_visibility, names='value')

def run_transformation(_):
    """Runs the appropriate transformation based on user selection."""

    input_folder = input_folder_widget.value
    data_type = data_type_selection.value
    tracking_format = tracking_format_selection.value

    if not os.path.exists(input_folder):
        with output_log:
            print("❌ Error: Input folder does not exist.")
        return

    if data_type == "User provided points":
        transform_points_csv(input_folder)
    elif data_type == "Segmentation mask":
        process_segmentation_folder(input_folder)
    else:
        with output_log:
            print(f"🔴 Running Tracking Result Transformation for {tracking_format}...")
        if tracking_format == "TrackMate (CSV)":
            transform_trackmate_res(input_folder, "csv")
        elif tracking_format == "TrackMate (XML)":
            transform_trackmate_res(input_folder, "xml")
        elif tracking_format == "Mastodon/Elephant":
            transform_mastodon_csv(input_folder)
        else:
            transform_ultrack_csv(input_folder)
    with output_log:
        print("✅ Transformation completed successfully!")
run_button.on_click(run_transformation)  # Connecte le bouton à la fonction

# Display UI
display(ui)